In [30]:
#pip install langchain langgraph langchain-community langchain-text-splitters faiss-cpu pandas python-dotenv

In [31]:
#ollama pull llama3

In [32]:
import re

In [33]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) 

In [34]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(
    model="llama3",
    temperature=0.2,        
)

In [35]:
from langchain_core.messages import HumanMessage

In [36]:
from langchain_core.prompts import ChatPromptTemplate

router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are TechFlow Router.\n"
     "Return ONLY valid JSON. No extra text.\n"
     "Format EXACTLY:\n"
     "{{"
     "\"intent\":\"cancel|tech_support|billing|unknown\","
     "\"email\":null,"
     "\"reason\":null"
     "}}\n"
     "Rules:\n"
     "- Cancel Care+/insurance/plan => intent=cancel\n"
     "- Phone problems => intent=tech_support\n"
     "- Billing/charges => intent=billing\n"
     "- Otherwise => unknown\n"
    ),
    ("human", "{user_message}")
])

In [37]:
print("router_prompt exists?", "router_prompt" in globals())#

router_prompt exists? True


# Create Shared Memory

In [38]:
def make_state(user_message: str) -> dict:
    return {
        "messages": [HumanMessage(content=user_message)],
        "intent": "unknown",
        "email": None,
        "customer_id": None,
        "customer_profile": None,
        "cancel_reason": None,
        "offer": None,
        "cancel_confirmed": False,
        "rag_context": None,
        "sources": None,
        "error": None,
    }

# Tools

In [39]:
from langchain_core.tools import tool
import csv
import json
from datetime import datetime
from pathlib import Path

In [40]:
@tool
def get_customer_data(email: str) -> dict:
    """Load customer profile from data/customers.csv by email."""
    csv_path = Path("data") / "customers.csv"
    if not csv_path.exists():
        return {"error": "missing_file", "file": str(csv_path)}

    target = email.strip().lower()

    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row_email = (row.get("email") or "").strip().lower()
            if row_email == target:
                return row

    return {"error": "customer_not_found", "email": email}

In [41]:
@tool
def calculate_retention_offer(customer_tier: str, reason: str) -> dict:
    """Generate retention offer using data/retention_rules.json"""

    json_path = Path("data") / "retention_rules.json"
    if not json_path.exists():
        return {"error": "missing_file", "file": str(json_path)}

    rules = json.loads(json_path.read_text(encoding="utf-8"))

    tier = (customer_tier or "default").lower()
    tier_rules = rules.get(tier, rules.get("default", {}))

    reason_lower = (reason or "").lower()

    # Basic classification logic
    if any(word in reason_lower for word in ["afford", "expensive", "money", "cost"]):
        offer = tier_rules.get("financial_hardship")
    else:
        offer = tier_rules.get("general")

    if not offer:
        return {"error": "no_offer_found", "tier": tier}

    return offer

In [42]:
@tool
def update_customer_status(customer_id: str, action: str) -> dict:
    """Process cancellations/changes by logging to outputs/status_log.txt"""
    log_path = Path("outputs") / "status_log.txt"

    ts = datetime.utcnow().isoformat() + "Z"
    line = f"{ts} | customer_id={customer_id} | action={action}\n"

    try:
        with log_path.open("a", encoding="utf-8") as f:
            f.write(line)
    except Exception as e:
        return {"error": "log_write_failed", "details": str(e)}

    return {"ok": True, "customer_id": customer_id, "action": action, "logged_to": str(log_path)}

# RAG

In [43]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

In [44]:
from pathlib import Path

POLICY_DIR = Path("data") / "Policy Documents"

def load_policy_docs():
    docs = []
    for fp in POLICY_DIR.glob("*.md"):
        docs.extend(TextLoader(str(fp), encoding="utf-8").load())
    return docs

policy_docs = load_policy_docs()
print("Loaded docs:", len(policy_docs))

splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
policy_chunks = splitter.split_documents(policy_docs)
print("Chunks:", len(policy_chunks))

emb = OllamaEmbeddings(model="nomic-embed-text")
policy_db = FAISS.from_documents(policy_chunks, emb)

retriever = policy_db.as_retriever(search_kwargs={"k": 4})

Loaded docs: 3
Chunks: 13


# Agents

In [45]:
import json
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage 

In [46]:
def agent1_orchestrator(state: dict) -> dict:
    user_message = state["messages"][-1].content

    # Route using LLM
    prompt_messages = router_prompt.invoke({"user_message": user_message})
    raw_text = llm.invoke(prompt_messages).content

    try:
        data = parse_router_json(raw_text)
    except Exception:
        data = {"intent": "unknown", "email": None, "reason": None}

    # Keep intent/reason if router returns null on later turns
    new_intent = data.get("intent")
    if new_intent and new_intent != "unknown":
        state["intent"] = new_intent

    new_reason = data.get("reason")
    if new_reason:
        state["cancel_reason"] = new_reason

    # Email extraction
    match = re.search(r"\S+@\S+\.\S+", user_message)
    if match:
        state["email"] = match.group(0)
    else:
        state["email"] = data.get("email") or state.get("email")

    # Customer lookup
    if state.get("email"):
        customer = get_customer_data.invoke({"email": state["email"]})

        if customer.get("error"):
            state["error"] = customer["error"]
            state["customer_profile"] = None
            state["customer_id"] = None
        else:
            state["error"] = None
            state["customer_profile"] = customer
            state["customer_id"] = customer.get("customer_id")

    # RAG context
    docs = retriever.invoke(user_message)
    state["rag_context"] = "\n\n".join(d.page_content for d in docs)
    state["sources"] = [d.metadata for d in docs]

    return state

In [47]:
problem_solver_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are TechFlow Problem Solver Agent.\n"
     "You must help the customer based on the intent.\n"
     "- If intent=cancel: be empathetic, explain value, propose offer, ask: "
     "- If intent=tech_support: give troubleshooting steps.\n"
     "- If intent=billing: explain likely reasons and next steps.\n"
     "Use POLICY CONTEXT if relevant.\n"
     "Be clear and beginner-friendly."
    ),
    ("human",
     "Intent: {intent}\n"
     "Customer Profile: {profile}\n"
     "Cancel Reason: {reason}\n"
     "Offer: {offer}\n"
     "\nPOLICY CONTEXT:\n{context}\n"
     "\nUser Message: {user_message}"
    )
])

def agent2_problem_solver(state: dict) -> dict:
    intent = state.get("intent", "unknown")
    profile = state.get("customer_profile") or {}
    reason = state.get("cancel_reason") or ""
    context = state.get("rag_context") or ""
    user_message = state["messages"][-1].content

    offer = None
    if intent == "cancel":
        tier = profile.get("customer_tier") or profile.get("tier") or "default"
        offer = calculate_retention_offer.invoke({"customer_tier": str(tier), "reason": str(reason)})
        state["offer"] = offer

    prompt_messages = problem_solver_prompt.invoke({
        "intent": intent,
        "profile": profile,
        "reason": reason,
        "offer": offer,
        "context": context,
        "user_message": user_message
    })

    response = llm.invoke(prompt_messages).content
    state["messages"].append(AIMessage(content=response))
    return state

In [48]:
def agent3_processor(state: dict) -> dict:
    customer_id = state.get("customer_id")
    if not customer_id:
        state["messages"].append(AIMessage(content="I can’t process changes until I can identify your account (please share your email)."))
        return state

    result = update_customer_status.invoke({"customer_id": customer_id, "action": "cancel_care_plus"})
    if result.get("error"):
        state["messages"].append(AIMessage(content="Sorry — I couldn’t process the cancellation due to a system error. Please try again."))
    else:
        state["messages"].append(AIMessage(content="Done — your TechFlow has been cancelled. You’ll receive confirmation shortly."))
    return state

In [49]:
def run_chat(user_message: str) -> dict:
    state = make_state(user_message)

    # Agent 1: Orchestrator
    state = agent1_orchestrator(state)

    # Detect confirmation
    state = detect_cancel_confirmation(state)

    # Agent 2: Problem Solver
    state = agent2_problem_solver(state)

    # Agent 3: Processor (only if confirmed cancel)
    if state.get("intent") == "cancel" and state.get("cancel_confirmed") is True:
        state = agent3_processor(state)

    return state

In [50]:
def detect_cancel_confirmation(state: dict) -> dict:
    # Only allow confirmation if intent is cancel
    if state.get("intent") != "cancel":
        state["cancel_confirmed"] = False
        return state

    text = state["messages"][-1].content.lower()

    confirm_phrases = [
        "yes",
        "confirm",
        "cancel it",
        "go ahead",
        "please cancel",
        "do it",
        "proceed"
    ]

    state["cancel_confirmed"] = any(p in text for p in confirm_phrases)

    return state

In [51]:
result = run_chat("Yes cancel it. My email is sarah.chen@email.com")
print(result["messages"][-1].content)

I'm so sorry to hear that you're considering cancelling your subscription.

Before we proceed with the cancellation process, I'd like to highlight the value you've received from our services. As a Care+ Premium member, you've enjoyed priority support, 24/7 availability, and proactive monitoring, which has helped keep your device running smoothly. You've also had access to additional perks like travel coverage, family sharing, upgrade protection, and cloud backup.

If you're still interested in cancelling, I'd like to offer a special incentive to make the transition smoother. We can provide a partial refund of $50 towards your next subscription or purchase with us. This is a one-time offer, exclusive to customers who are cancelling their subscription.

Before we proceed with the cancellation, may I ask what's driving your decision? Is there something specific that's not meeting your expectations?

Please let me know, and I'll guide you through the cancellation process accordingly.


# Multi-turn Chat Loop

In [52]:
def chat_turn(state: dict, user_message: str) -> dict:
    # 1) Add new user message to history
    state["messages"].append(HumanMessage(content=user_message))

    # 2) Agent 1: route + customer lookup + RAG
    state = agent1_orchestrator(state)

    # 3) Detect confirmation (only for cancel intent)
    state = detect_cancel_confirmation(state)

    # 4) Agent 2: respond
    state = agent2_problem_solver(state)

    # 5) Agent 3: process only if confirmed cancel
    if state.get("intent") == "cancel" and state.get("cancel_confirmed") is True:
        state = agent3_processor(state)

    return state

In [53]:
state = make_state("Hi")   # start state with a first message (optional)

In [54]:
state = make_state("")     # start empty
state["messages"] = []     # clear initial message

In [55]:
state = make_state("")
state["messages"] = []

state = chat_turn(state, "hey can't afford care+ anymore, need to cancel")
print(state["messages"][-1].content)

state = chat_turn(state, "My email is sarah.chen@email.com")
print(state["messages"][-1].content)

state = chat_turn(state, "yes cancel it")
print(state["messages"][-1].content)

I understand that you're facing a tough decision and need to cancel your Care+ subscription due to financial constraints. I want to acknowledge the value that Care+ has provided for you in the past.

Before we proceed with the cancellation, I'd like to highlight some of the benefits you've enjoyed as a Care+ customer:

* Priority support access: You've had direct specialist access and callback scheduling for complex issues.
* Preventive care: Regular device health checkups, software optimization, and early warning for potential hardware issues have helped you stay ahead of any potential problems.

I'd like to offer an alternative solution that might help you continue enjoying these benefits at a more affordable price point. Would you be interested in exploring our Care+ Basic plan, which offers core protection features like screen repair with a $29 deductible and standard warranty extension coverage? This plan is priced at just $6.99/month.

If you're not ready to commit to another mon

# Run the 5 Required Test Conversations

In [56]:
def run_conversation_test(title: str, turns: list[str]) -> dict:
    print("\n" + "="*70)
    print(title)
    print("="*70)

    state = make_state("")
    state["messages"] = []  # start clean

    for i, user_msg in enumerate(turns, 1):
        state = chat_turn(state, user_msg)
        assistant_reply = state["messages"][-1].content
        print(f"\nTurn {i} (User): {user_msg}")
        print(f"Turn {i} (AI):   {assistant_reply}")

    print("\n--- Final State ---")
    print("intent:", state.get("intent"))
    print("email:", state.get("email"))
    print("customer_id:", state.get("customer_id"))
    print("cancel_confirmed:", state.get("cancel_confirmed"))
    print("error:", state.get("error"))

    return state

In [62]:
run_conversation_test(
    "TEST 1: Money Problems",
    [
        "hey can't afford the $13/month anymore, need to cancel",
        "my email is sarah.chen@email.com",
    ]
)



TEST 1: Money Problems

Turn 1 (User): hey can't afford the $13/month anymore, need to cancel
Turn 1 (AI):   I understand that you're facing a tough decision to cancel your TechFlow Care+ subscription due to financial constraints. I want to acknowledge that it's not an easy choice, and I'm here to help.

Before we proceed with cancellation, let me highlight the value of having Care+. You've been enjoying benefits like accident protection, device services, and priority support, which have likely provided you with peace of mind and saved you money in the long run. According to our statistics, 73% of customers use their benefits within the first 18 months, and the average claim value is $280 (screen repairs).

If you're willing to consider an alternative, I'd like to propose a more affordable option: Care+ Basic ($6.99/month). This plan still offers core protection for your device, including screen repair with a $29 deductible, hardware defects coverage, and basic support.

However, if y

{'messages': [HumanMessage(content="hey can't afford the $13/month anymore, need to cancel", additional_kwargs={}, response_metadata={}),
  AIMessage(content="I understand that you're facing a tough decision to cancel your TechFlow Care+ subscription due to financial constraints. I want to acknowledge that it's not an easy choice, and I'm here to help.\n\nBefore we proceed with cancellation, let me highlight the value of having Care+. You've been enjoying benefits like accident protection, device services, and priority support, which have likely provided you with peace of mind and saved you money in the long run. According to our statistics, 73% of customers use their benefits within the first 18 months, and the average claim value is $280 (screen repairs).\n\nIf you're willing to consider an alternative, I'd like to propose a more affordable option: Care+ Basic ($6.99/month). This plan still offers core protection for your device, including screen repair with a $29 deductible, hardwar

In [58]:
run_conversation_test(
    "TEST 2: Phone Problems (overheating)",
    [
        "this phone keeps overheating, want to return it and cancel everything",
        "my email is robert.taylor@email.com"
    ]
)


TEST 2: Phone Problems (overheating)

Turn 1 (User): this phone keeps overheating, want to return it and cancel everything
Turn 1 (AI):   I'm so sorry to hear that your phone is overheating. That can be really frustrating!

Before we discuss returning the device, I'd like to offer some troubleshooting steps from our TechFlow Device Troubleshooting Guide.

Have you tried any of these quick fixes:

* Remove phone case and let device cool down
* Close all background apps
* Reduce screen brightness temporarily
* Avoid direct sunlight and heat sources

If none of those work, we can explore software solutions or even consider replacing the device if it's consistently overheating above 104°F (40°C) during normal use.

Regarding returning the phone, our Standard Return Policy states that unopened devices have a 30-day return window from the purchase date. Since your phone is still within this timeframe, you're eligible to return it.

As a valued customer, I'd like to propose an offer: if you 

{'messages': [HumanMessage(content='this phone keeps overheating, want to return it and cancel everything', additional_kwargs={}, response_metadata={}),
  AIMessage(content="I'm so sorry to hear that your phone is overheating. That can be really frustrating!\n\nBefore we discuss returning the device, I'd like to offer some troubleshooting steps from our TechFlow Device Troubleshooting Guide.\n\nHave you tried any of these quick fixes:\n\n* Remove phone case and let device cool down\n* Close all background apps\n* Reduce screen brightness temporarily\n* Avoid direct sunlight and heat sources\n\nIf none of those work, we can explore software solutions or even consider replacing the device if it's consistently overheating above 104°F (40°C) during normal use.\n\nRegarding returning the phone, our Standard Return Policy states that unopened devices have a 30-day return window from the purchase date. Since your phone is still within this timeframe, you're eligible to return it.\n\nAs a valu

In [ ]:
run_conversation_test(
    "TEST 3: Questioning Value",
    [
        "paying for care+ but never used it, maybe just get rid of it?",
        "my email is mike.rodriguez@email.com"
    ]
)


TEST 3: Questioning Value

Turn 1 (User): paying for care+ but never used it, maybe just get rid of it?
Turn 1 (AI):   I understand your concern about not using the Care+ service and wondering if you should cancel it. I want to acknowledge that it's completely normal to reevaluate your subscriptions and services from time to time.

Before we discuss cancellation, let me highlight the value you're getting with Care+. As a customer, you're already protected by our Core Protection features, including Screen Repair, Hardware Defects, and Basic Support. These benefits can provide peace of mind in case something happens to your device.

I'd like to propose an alternative: instead of canceling, why not take advantage of the benefits you've already paid for? You might be surprised at how quickly you'll need to use one of these features. According to our statistics, 73% of customers use their benefits within the first 18 months.

If you're still unsure, I can offer a special deal to help you g

{'messages': [HumanMessage(content='paying for care+ but never used it, maybe just get rid of it?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="I understand your concern about not using the Care+ service and wondering if you should cancel it. I want to acknowledge that it's completely normal to reevaluate your subscriptions and services from time to time.\n\nBefore we discuss cancellation, let me highlight the value you're getting with Care+. As a customer, you're already protected by our Core Protection features, including Screen Repair, Hardware Defects, and Basic Support. These benefits can provide peace of mind in case something happens to your device.\n\nI'd like to propose an alternative: instead of canceling, why not take advantage of the benefits you've already paid for? You might be surprised at how quickly you'll need to use one of these features. According to our statistics, 73% of customers use their benefits within the first 18 months.\n\nIf you're st

In [60]:
run_conversation_test(
    "TEST 4: Technical Help Needed",
    [
        "my phone won't charge anymore, tried different cables",
        "my email is lisa.kim@email.com"
    ]
)


TEST 4: Technical Help Needed

Turn 1 (User): my phone won't charge anymore, tried different cables
Turn 1 (AI):   I'd be happy to help you troubleshoot the issue with your phone not charging. 

Let's start by going through the steps in our Charging Problems policy context:

**Step 1 - Cable and Adapter**:
- Try different USB cable
- Test with different wall adapter
- Check charging port for debris or damage
- Clean charging port with dry brush

Have you tried any of these steps already?

Turn 2 (User): my email is lisa.kim@email.com
Turn 2 (AI):   Hello Lisa! I'm happy to help you with your concern.

Since we're not sure what your intent is, let's start by exploring the options together. You can choose from the following:

1. **Cancel**: If you'd like to cancel your subscription or return a product.
2. **Tech Support**: If you need assistance with troubleshooting an issue or setting up a device.
3. **Billing**: If you have questions about your monthly charge, payment method, or billi

{'messages': [HumanMessage(content="my phone won't charge anymore, tried different cables", additional_kwargs={}, response_metadata={}),
  AIMessage(content="I'd be happy to help you troubleshoot the issue with your phone not charging. \n\nLet's start by going through the steps in our Charging Problems policy context:\n\n**Step 1 - Cable and Adapter**:\n- Try different USB cable\n- Test with different wall adapter\n- Check charging port for debris or damage\n- Clean charging port with dry brush\n\nHave you tried any of these steps already?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='my email is lisa.kim@email.com', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello Lisa! I'm happy to help you with your concern.\n\nSince we're not sure what your intent is, let's start by exploring the options together. You can choose from the following:\n\n1. **Cancel**: If you'd like to cancel your subscription or ret

In [61]:
run_conversation_test(
    "TEST 5: Billing Question",
    [
        "got charged $15.99 but thought care+ was $12.99, what's the extra?",
        "my email is james.wilson@email.com"
    ]
)


TEST 5: Billing Question

Turn 1 (User): got charged $15.99 but thought care+ was $12.99, what's the extra?
Turn 1 (AI):   I'm happy to help you with that!

It looks like there might have been a small mistake on your Care+ bill. The standard rate for Care+ Basic is indeed $6.99/month, not $15.99.

Could you please double-check your account information or recent transactions to see if there was an error or any additional services added? I'm here to help you figure out what might have caused the discrepancy.

In the meantime, let me remind you that Care+ Basic offers some great benefits, including Screen Repair with a $29 deductible, Hardware Defects coverage, and Basic Support during business hours. You can also take advantage of Mail-in Repair services if needed.

If you're interested in upgrading to Care+ Premium, which includes additional features like Accident Protection, Device Services, and Family Benefits, I'd be happy to walk you through the benefits and costs.

What would you 

{'messages': [HumanMessage(content="got charged $15.99 but thought care+ was $12.99, what's the extra?", additional_kwargs={}, response_metadata={}),
  AIMessage(content="I'm happy to help you with that!\n\nIt looks like there might have been a small mistake on your Care+ bill. The standard rate for Care+ Basic is indeed $6.99/month, not $15.99.\n\nCould you please double-check your account information or recent transactions to see if there was an error or any additional services added? I'm here to help you figure out what might have caused the discrepancy.\n\nIn the meantime, let me remind you that Care+ Basic offers some great benefits, including Screen Repair with a $29 deductible, Hardware Defects coverage, and Basic Support during business hours. You can also take advantage of Mail-in Repair services if needed.\n\nIf you're interested in upgrading to Care+ Premium, which includes additional features like Accident Protection, Device Services, and Family Benefits, I'd be happy to wa